# 1. Análisis de Distribución Salarial por Departamento y Género

**Objetivo:** Analizar cómo se distribuyen los salarios entre los diferentes departamentos y cómo varía esta distribución en función del género para identificar posibles brechas salariales.

In [ ]:
import pandas as pd
import psycopg2
import getpass
import matplotlib.pyplot as plt
import seaborn as sns

### Conexión a la Base de Datos
Por favor, introduce las credenciales de la base de datos a continuación.

In [ ]:
db_name = input("Database Name: ")
db_user = input("Database User: ")
db_pass = getpass.getpass("Database Password: ")
db_host = "postgres"
db_port = "5432"

try:
    conn = psycopg2.connect(
        dbname=db_name,
        user=db_user,
        password=db_pass,
        host=db_host,
        port=db_port
    )
    print("Conexión a la base de datos exitosa.")
except psycopg2.OperationalError as e:
    print(f"Error en la conexión: {e}")

### (Debug) Verificar datos de salarios y departamentos actuales

In [ ]:
debug_query_salaries = "SELECT COUNT(*) FROM salaries WHERE to_date = '9999-12-31';"
debug_query_dept_emp = "SELECT COUNT(*) FROM dept_emp WHERE to_date = '9999-12-31';"

try:
    salaries_count = pd.read_sql_query(debug_query_salaries, conn).iloc[0,0]
    dept_emp_count = pd.read_sql_query(debug_query_dept_emp, conn).iloc[0,0]
    print(f"Registros en 'salaries' con to_date='9999-12-31': {salaries_count}")
    print(f"Registros en 'dept_emp' con to_date='9999-12-31': {dept_emp_count}")
except Exception as e:
    print(f"Error durante la depuración: {e}")

### Consulta SQL para obtener los datos

In [ ]:
query = """
SELECT 
    d.dept_name,
    e.gender,
    s.salary
FROM 
    employees e
JOIN 
    salaries s ON e.emp_no = s.emp_no
JOIN 
    dept_emp de ON e.emp_no = de.emp_no
JOIN 
    departments d ON de.dept_no = d.dept_no
WHERE 
    s.to_date = '9999-12-31' -- Salario actual
    AND de.to_date = '9999-12-31'; -- Departamento actual
"""

try:
    df = pd.read_sql_query(query, conn)
    print("DataFrame de distribución salarial:")
    display(df.head())
except Exception as e:
    print(f"Error al ejecutar la consulta: {e}")

### Análisis y Visualización

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

if 'df' in locals() and not df.empty:
    # Gráfico de distribución de salarios por departamento
    plt.figure(figsize=(12, 8))
    sns.boxplot(data=df, x='dept_name', y='salary', palette='viridis')
    plt.title('Distribución de Salarios por Departamento', fontsize=16)
    plt.xlabel('Departamento', fontsize=12)
    plt.ylabel('Salario', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Gráfico de distribución de salarios por género
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='gender', y='salary', palette='plasma')
    plt.title('Distribución de Salarios por Género', fontsize=16)
    plt.xlabel('Género', fontsize=12)
    plt.ylabel('Salario', fontsize=12)
    plt.show()

    # Gráfico de distribución de salarios por departamento y género
    plt.figure(figsize=(14, 9))
    sns.boxplot(data=df, x='dept_name', y='salary', hue='gender', palette='magma')
    plt.title('Distribución de Salarios por Departamento y Género', fontsize=16)
    plt.xlabel('Departamento', fontsize=12)
    plt.ylabel('Salario', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Género')
    plt.tight_layout()
    plt.show()

else:
    print("El DataFrame está vacío. No se pueden generar los gráficos.")

# Cerrar la conexión
if 'conn' in locals() and conn is not None:
    conn.close()
    print("Conexión cerrada.")